# Onshape Pack and Go
Exports all released parts (STEP) and their linked drawings (PDF) from an assembly.

**On Google Colab:**
1. Click the 🔑 **Secrets** tab in the left sidebar
2. Add `ONSHAPE_ACCESS_KEY` and `ONSHAPE_SECRET_KEY` (from [Onshape Developer Portal](https://dev-portal.onshape.com/))
3. Paste your assembly URL in the Config cell below
4. Runtime → Run all

**Running locally:**
1. Fill in your keys in the `.env` file next to this notebook
2. Paste your assembly URL in the Config cell below
3. Run all cells

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install requests python-dotenv --quiet

In [ ]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
ASSEMBLY_URL = input("Paste assembly URL: ").strip()

In [ ]:
# ── 3. Load API keys ──────────────────────────────────────────────────────────
import os

try:
    from google.colab import userdata
    ACCESS_KEY = userdata.get('ONSHAPE_ACCESS_KEY')
    SECRET_KEY = userdata.get('ONSHAPE_SECRET_KEY')
    print("\u2713 API keys loaded from Colab Secrets")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    ACCESS_KEY = os.environ.get('ONSHAPE_ACCESS_KEY', '')
    SECRET_KEY = os.environ.get('ONSHAPE_SECRET_KEY', '')
    if ACCESS_KEY:
        print("\u2713 API keys loaded from .env file")
    else:
        print("Keys not found in .env")

if not ACCESS_KEY or not SECRET_KEY:
    raise ValueError("API keys not found. Add ONSHAPE_ACCESS_KEY and ONSHAPE_SECRET_KEY to Colab Secrets or a .env file.")

In [ ]:
# ── 4. API helpers ────────────────────────────────────────────────────────────
import requests
import time
import re
import io
import zipfile

BASE_URL = "https://cad.onshape.com/api/v6"
AUTH     = (ACCESS_KEY, SECRET_KEY)
HEADERS  = {"Accept": "application/json"}
POLL_INTERVAL = 3
POLL_TIMEOUT  = 300


def api_get(path, query=None):
    resp = requests.get(BASE_URL + path, auth=AUTH, headers=HEADERS, params=query)
    resp.raise_for_status()
    return resp.json()


def api_post(path, body=None, query=None):
    resp = requests.post(BASE_URL + path, auth=AUTH,
                         headers={**HEADERS, "Content-Type": "application/json"},
                         params=query, json=body)
    resp.raise_for_status()
    return resp.json()


def api_get_binary(path, query=None):
    resp = requests.get(BASE_URL + path, auth=AUTH,
                        headers={"Accept": "application/octet-stream"},
                        params=query)
    resp.raise_for_status()
    return resp.content


def poll_translation(translation_id, doc_id):
    deadline = time.time() + POLL_TIMEOUT
    while time.time() < deadline:
        status = api_get(f"/translations/{translation_id}")
        state  = status.get("requestState", "")
        if state == "DONE":
            ext_ids = status.get("resultExternalDataIds") or []
            if not ext_ids:
                raise RuntimeError(f"Translation {translation_id} done but no files returned.")
            return api_get_binary(f"/documents/d/{doc_id}/externaldata/{ext_ids[0]}")
        elif state == "FAILED":
            raise RuntimeError(f"Translation failed: {status.get('failureReason')}")
        time.sleep(POLL_INTERVAL)
    raise TimeoutError(f"Translation {translation_id} timed out after {POLL_TIMEOUT}s")


print("\u2713 API helpers ready")

In [ ]:
# ── 5. Parse URL and get assembly name ───────────────────────────────────────

def parse_url(url):
    m = re.search(r"documents/([a-f0-9]+)/(w|v|m)/([a-f0-9]+)/e/([a-f0-9]+)", url)
    if not m:
        raise ValueError(f"Could not parse Onshape URL: {url}")
    return m.group(1), m.group(2), m.group(3), m.group(4)


def get_assembly_name(did, wvm, wvmid, eid):
    try:
        elements = api_get(f"/documents/d/{did}/{wvm}/{wvmid}/elements", query={"elementId": eid})
        return elements[0].get("name", "onshape_assembly") if elements else "onshape_assembly"
    except Exception:
        return "onshape_assembly"


did, wvm, wvmid, eid = parse_url(ASSEMBLY_URL)
assembly_name = get_assembly_name(did, wvm, wvmid, eid)
folder_name   = re.sub(r'[^\w\-.]', '_', assembly_name)

print(f"  Document:  {did}")
print(f"  Workspace: {wvmid} ({wvm})")
print(f"  Element:   {eid}")
print(f"  Assembly:  {assembly_name}")

In [ ]:
# ── 6. Get released parts from BOM ───────────────────────────────────────────

def get_released_parts(did, wvm, wvmid, eid):
    data = api_get(f"/assemblies/d/{did}/{wvm}/{wvmid}/e/{eid}/bom",
                   query={"bomType": "flattened", "indented": "false", "multiLevel": "false"})

    headers   = {h["propertyName"]: h["id"] for h in data.get("headers", [])}
    name_col  = headers.get("name")
    state_col = headers.get("state")
    pn_col    = headers.get("partNumber")
    rev_col   = headers.get("revision")

    if not state_col:
        print("  \u26a0 No 'State' column in BOM \u2014 release filtering skipped.")

    seen, parts, skipped = set(), [], []

    for row in data.get("rows", []):
        vals  = row.get("headerIdToValue", {})
        name  = vals.get(name_col) or "unnamed"
        state = vals.get(state_col) or ""

        if state_col and state.lower() != "released":
            skipped.append(f"{name} ({state or 'no state'})")
            continue

        src = row.get("itemSource", {})
        if not src:
            continue

        part_id    = src.get("partId")
        element_id = src.get("elementId")
        doc_id     = src.get("documentId") or did
        key        = (doc_id, element_id, part_id)

        if key in seen:
            continue
        seen.add(key)
        parts.append({
            "name":       name,
            "partId":     part_id,
            "elementId":  element_id,
            "documentId": doc_id,
            "wvmId":      src.get("wvmId", wvmid),
            "wvmType":    src.get("wvmType", wvm),
            "partNumber": vals.get(pn_col) or "",
            "revision":   vals.get(rev_col) or "",
        })

    if skipped:
        print(f"  Skipped {len(skipped)} non-released part(s):")
        for s in skipped:
            print(f"    \u2717 {s}")

    return parts


print("Fetching released parts from BOM...")
parts = get_released_parts(did, wvm, wvmid, eid)
print(f"  Found {len(parts)} released part(s):")
for p in parts:
    print(f"    \u2022 {p['name']} (pn={p['partNumber']}, rev={p['revision']})")

if not parts:
    raise SystemExit("No released parts found.")

In [ ]:
# ── 7. Export parts as STEP ───────────────────────────────────────────────────
import os

os.makedirs(folder_name, exist_ok=True)

def export_step(part):
    result = api_post(
        f"/partstudios/d/{part['documentId']}/{part['wvmType']}/{part['wvmId']}/e/{part['elementId']}/translations",
        body={"formatName": "STEP", "partIds": part["partId"], "storeInDocument": False},
    )
    return poll_translation(result["id"], part["documentId"])


print("Exporting STEP files...")
step_files = {}
step_base  = {}  # (partNumber, revision) -> base filename without extension
for part in parts:
    print(f"  {part['name']} ...", end=" ", flush=True)
    try:
        data      = export_step(part)
        part_safe = re.sub(r'[^\w\-.]', '_', part['name'])
        fname     = f"{part['partNumber']}-{part['revision']}-{part_safe}.step"
        with open(os.path.join(folder_name, fname), "wb") as f:
            f.write(data)
        step_files[fname] = data
        step_base[(part['partNumber'], part['revision'])] = fname[:-5]
        print("\u2713")
    except Exception as e:
        print(f"\u2717 ({e})")

print(f"\nExported {len(step_files)} STEP file(s) to {folder_name}/")


In [ ]:
# ── 8. Find released drawings (scans all document versions) ───────────────────

def get_workspace(doc_id):
    try:
        return api_get(f"/documents/{doc_id}").get("defaultWorkspace", {}).get("id", "")
    except Exception:
        return ""


def get_drawing_metadata(did, wvm, wvmid, eid):
    try:
        data  = api_get(f"/metadata/d/{did}/{wvm}/{wvmid}/e/{eid}")
        props = {p["name"]: p.get("value") for p in data.get("properties", [])}
        return props.get("Part number") or "", props.get("Revision") or "", props.get("State") or ""
    except Exception:
        return "", "", ""


def _normalize_drawing_name(name):
    """Strip trailing drawing suffixes (e.g. 'Drawing 1', 'DWG 2') for name matching."""
    name = re.sub(r'[\s\-_]+(drawing|dwg|drw)[\s\-_]*\d*\s*$', '', name, flags=re.IGNORECASE)
    return name.strip().lower()


def find_released_drawings(parts, did):
    ws_id = get_workspace(did)
    if not ws_id:
        return []

    try:
        elements = api_get(f"/documents/d/{did}/w/{ws_id}/elements")
    except Exception:
        return []

    all_drawing_els = {
        el["id"]: el.get("name", el["id"])
        for el in elements
        if el.get("elementType") == "APPLICATION"
        and el.get("dataType") == "onshape-app/drawing"
    }
    print(f"  Found {len(all_drawing_els)} drawing(s) in document.")

    # Pre-filter by name: only scan drawings whose name (minus drawing suffix)
    # matches a part name. Avoids expensive version scans for unrelated drawings.
    part_names = {p["name"].strip().lower() for p in parts}
    drawing_els = {
        eid: name for eid, name in all_drawing_els.items()
        if _normalize_drawing_name(name) in part_names
    }

    # If any part has no name-matched candidate, fall back to scanning all drawings
    # so we don't miss drawings with non-standard naming conventions.
    matched_names = {_normalize_drawing_name(name) for name in drawing_els.values()}
    unmatched_parts = [p for p in parts if p["name"].strip().lower() not in matched_names]
    if unmatched_parts:
        print(f"  \u26a0 No name-matched drawing for: {', '.join(p['name'] for p in unmatched_parts)}")
        print(f"  Falling back to scanning all {len(all_drawing_els)} drawing(s).")
        drawing_els = all_drawing_els
    else:
        skipped_count = len(all_drawing_els) - len(drawing_els)
        if skipped_count:
            print(f"  Pre-filtered to {len(drawing_els)} candidate(s) by name "
                  f"({skipped_count} drawing(s) don't match any part name).")

    pn_rev_lookup  = {(p["partNumber"], p["revision"]): p for p in parts if p["partNumber"]}
    pn_only_lookup = {p["partNumber"] for p in parts if p["partNumber"]}

    try:
        versions = api_get(f"/documents/{did}/versions")
    except Exception:
        versions = []

    found     = {}
    done      = set()
    wrong_rev = {}

    for ver in versions:
        vid        = ver["id"]
        unresolved = [eid for eid in drawing_els if eid not in done]
        if not unresolved:
            break
        for eid in unresolved:
            pn, rev, state = get_drawing_metadata(did, "v", vid, eid)
            if state != "2":
                continue
            name = drawing_els[eid]
            if pn not in pn_only_lookup:
                done.add(eid)
                if pn:
                    print(f"    \u2717 '{name}' skipped (pn={pn!r} not in released BOM)")
            elif (pn, rev) in pn_rev_lookup:
                part = pn_rev_lookup[(pn, rev)]
                print(f"    \u2713 '{name}' (pn={pn}, rev={rev}) \u2192 '{part['name']}'")
                found[eid] = {"id": eid, "name": name, "documentId": did,
                              "wvm": "v", "wvmid": vid, "partNumber": pn, "revision": rev,
                              "partName": part["name"]}
                done.add(eid)
            else:
                if eid not in wrong_rev:
                    wrong_rev[eid] = (pn, rev)

    for eid, name in drawing_els.items():
        if eid not in found:
            if eid in wrong_rev:
                pn, rev = wrong_rev[eid]
                print(f"    \u2717 '{name}' skipped (pn={pn!r} released at rev={rev!r}, not in BOM)")
            elif eid not in done:
                print(f"    \u2717 '{name}' skipped (not released in any version)")

    matched_pn_revs = {(d["partNumber"], d["revision"]) for d in found.values()}
    for p in parts:
        if p["partNumber"] and (p["partNumber"], p["revision"]) not in matched_pn_revs:
            print(f"  \u26a0 No drawing found for '{p['name']}' (pn={p['partNumber']}, rev={p['revision']})")

    return list(found.values())


print("Finding linked drawings...")
docs = {}
for p in parts:
    p_did = p["documentId"]
    if p_did not in docs:
        docs[p_did] = {"wvm": p["wvmType"], "wvmid": p["wvmId"], "parts": []}
    docs[p_did]["parts"].append(p)

linked_drawings = []
for doc_did, info in docs.items():
    label = "assembly document" if doc_did == did else f"external document {doc_did}"
    print(f"  Scanning {label}...")
    linked_drawings += find_released_drawings(info["parts"], doc_did)

print(f"\n{len(linked_drawings)} drawing(s) matched.")

In [ ]:
# ── 9. Export drawings as PDF ─────────────────────────────────────────────────

def export_pdf(drawing):
    result = api_post(
        f"/drawings/d/{drawing['documentId']}/{drawing['wvm']}/{drawing['wvmid']}/e/{drawing['id']}/translations",
        body={"formatName": "PDF", "storeInDocument": False},
    )
    return poll_translation(result["id"], drawing["documentId"])


print("Exporting PDF drawings...")
pdf_files = {}
for drawing in linked_drawings:
    print(f"  {drawing['name']} ...", end=" ", flush=True)
    try:
        data  = export_pdf(drawing)
        base  = step_base.get((drawing['partNumber'], drawing['revision']))
        if base:
            fname = base + ".pdf"
        else:
            part_safe = re.sub(r'[^\w\-.]', '_', drawing['partName'])
            fname = f"{drawing['partNumber']}-{drawing['revision']}-{part_safe}.pdf"
        with open(os.path.join(folder_name, fname), "wb") as f:
            f.write(data)
        pdf_files[fname] = data
        print("\u2713")
    except Exception as e:
        print(f"\u2717 ({e})")

print(f"\nExported {len(pdf_files)} PDF file(s) to {folder_name}/")


In [ ]:
# ── 10. Package into ZIP and download ────────────────────────────────────────
zip_name = f"{folder_name}.zip"
buf      = io.BytesIO()

with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname, data in step_files.items():
        zf.writestr(f"{folder_name}/{fname}", data)
    for fname, data in pdf_files.items():
        zf.writestr(f"{folder_name}/{fname}", data)

zip_bytes = buf.getvalue()
print(f"ZIP: {zip_name} ({len(zip_bytes) / 1024:.1f} KB)")
print(f"  {len(step_files)} STEP file(s), {len(pdf_files)} PDF file(s)")

try:
    from google.colab import files
    with open(zip_name, "wb") as f:
        f.write(zip_bytes)
    files.download(zip_name)
    print(f"\n\u2713 Download started: {zip_name}")
except ImportError:
    with open(zip_name, "wb") as f:
        f.write(zip_bytes)
    print(f"\n\u2713 Saved locally: {zip_name}")